# T38 - 年报处理

## 概述

本项目专注于企业年报的自动化采集、OCR处理和财务信息提取。

### 核心功能
1. **年报采集**: 从企业预警通等平台爬取年报PDF
2. **OCR处理**: 使用百度OCR或PyMuPDF提取文本
3. **大模型提取**: 使用通义千问提取关键财务信息
4. **数据存储**: 将提取结果存入数据库

### 目录
1. [环境配置与依赖导入](#1-环境配置与依赖导入)
2. [数据库连接管理](#2-数据库连接管理)
3. [年报爬取模块](#3-年报爬取模块)
4. [PDF处理与OCR](#4-pdf处理与ocr)
5. [大模型信息提取](#5-大模型信息提取)
6. [数据处理与存储](#6-数据处理与存储)
7. [主流程执行](#7-主流程执行)

<a id='1-环境配置与依赖导入'></a>
## 1. 环境配置与依赖导入

导入所需的Python库和配置文件。

In [ ]:
# 标准库
import os
import sys
import re
import json
import time
import random
import shutil
from time import sleep
from pathlib import Path
from urllib import parse
from urllib.parse import urlparse, parse_qs
from datetime import datetime, date, timedelta

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import sqlalchemy
from sqlalchemy import text
from sqlalchemy.pool import NullPool

# 网络请求
import requests
import urllib.request

# PDF处理
import fitz  # PyMuPDF

# 大模型
from openai import OpenAI

# 本地配置
from config import (
    get_yq_connection_string,
    get_timinfo_connection_string,
    QYYJT_API_URL,
    QWEN_API_KEY,
    QWEN_BASE_URL,
    QWEN_MODEL,
    PDF_STORAGE_PATH,
    OCR_PENDING_PATH,
    ANNUAL_REPORT_CONFIG,
    DATABASE_TABLES,
    validate_config
)

print("依赖导入完成")
validate_config()

In [ ]:
# 配置常量
CONFIG = ANNUAL_REPORT_CONFIG
TABLES = DATABASE_TABLES

# 年报匹配正则
PATTERN_INCLUDE = re.compile(CONFIG['report_patterns']['include'])
PATTERN_EXCLUDE = re.compile(CONFIG['report_patterns']['exclude'])
PATTERN_YEAR = re.compile(CONFIG['report_patterns']['year'])

print("配置加载完成")
print(f"PDF存储路径: {PDF_STORAGE_PATH}")
print(f"OCR待处理路径: {OCR_PENDING_PATH}")

<a id='2-数据库连接管理'></a>
## 2. 数据库连接管理

建立与管理数据库连接，支持主数据库和备份数据库。

In [ ]:
class DatabaseManager:
    """数据库管理类"""
    
    def __init__(self):
        """初始化数据库连接"""
        self.yq_engine = sqlalchemy.create_engine(
            get_yq_connection_string(),
            poolclass=NullPool,
            pool_recycle=3600
        )
        
        self.timinfo_engine = sqlalchemy.create_engine(
            get_timinfo_connection_string(),
            poolclass=NullPool,
            pool_recycle=3600
        )
        
    def get_yq_connection(self):
        """获取yq数据库连接"""
        return self.yq_engine.begin()
    
    def get_timinfo_connection(self):
        """获取timinfo数据库连接"""
        return self.timinfo_engine.begin()
    
    def execute_query(self, sql, params=None, db='yq'):
        """执行查询语句"""
        engine = self.yq_engine if db == 'yq' else self.timinfo_engine
        with engine.begin() as conn:
            result = conn.execute(text(sql), params or {})
            return result.fetchall()
    
    def execute_update(self, sql, params=None, db='yq'):
        """执行更新语句"""
        engine = self.yq_engine if db == 'yq' else self.timinfo_engine
        with engine.begin() as conn:
            conn.execute(text(sql), params or {})
    
    def read_sql(self, sql, db='yq'):
        """读取SQL到DataFrame"""
        engine = self.yq_engine if db == 'yq' else self.timinfo_engine
        with engine.begin() as conn:
            return pd.read_sql(sql, conn)
    
    def to_sql(self, df, table_name, db='yq', if_exists='append'):
        """将DataFrame写入数据库"""
        engine = self.yq_engine if db == 'yq' else self.timinfo_engine
        with engine.begin() as conn:
            df.to_sql(table_name, conn, if_exists=if_exists, index=False)

# 初始化数据库管理器
db_manager = DatabaseManager()
print("数据库连接初始化完成")

In [ ]:
def query_annual_report_files():
    """查询待处理的年报文件列表"""
    sql = f"""
    SELECT A.trade_code, A.fileName, A.fileUrl, A.ocr,
           B.ths_issuer_name_cn_bond as company
    FROM {TABLES['annual_report_files']} A
    LEFT JOIN {TABLES['trade_code_temp']} B 
    ON A.trade_code = B.trade_code
    WHERE A.fileName != ''
    """
    return db_manager.read_sql(sql)

# 测试查询
df_files = query_annual_report_files()
print(f"待处理年报文件数量: {len(df_files)}")
df_files.head()

<a id='3-年报爬取模块'></a>
## 3. 年报爬取模块

从企业预警通爬取年报PDF文件。

In [ ]:
class AnnualReportCrawler:
    """年报爬取类"""
    
    def __init__(self, db_manager):
        self.db = db_manager
        self.base_url = QYYJT_API_URL
        self.headers = {
            'authority': 'www.qyyjt.cn',
            'scheme': 'https',
            'accept': '*/*',
            'accept-encoding': 'gzip, deflate, br, zstd',
            'accept-language': 'zh-CN,zh;q=0.9,en;q=0.8',
            'client': 'pc-web;pro',
            'content-type': 'application/x-www-form-urlencoded',
            'origin': 'https://www.qyyjt.cn',
            'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
    
    @staticmethod
    def sanitize_filename(filename):
        """清理文件名，使其符合Windows文件命名规范"""
        sanitized = re.sub(r'[<>:"/\\|?*]', '_', filename)
        return sanitized
    
    def search_company(self, company_name):
        """搜索公司获取公司代码"""
        encoded_name = parse.quote(company_name)
        url = f"{self.base_url}?root_type=securities&skip=0&pagesize=15&text={encoded_name}"
        
        data = {
            "root_type": "securities",
            "skip": 0,
            "pagesize": 15,
            "text": company_name
        }
        
        try:
            response = requests.post(
                url, 
                headers=self.headers, 
                data=parse.urlencode(data),
                timeout=30
            )
            result = response.json()
            
            if result.get('success') and result.get('data', {}).get('list'):
                return result['data']['list'][0]['code']
            return None
        except Exception as e:
            print(f"搜索公司失败: {company_name}, 错误: {e}")
            return None
    
    def get_annual_report_url(self, company_code, year='2023'):
        """获取年报URL"""
        url = f"{self.base_url}?exportFlag=false&aggs=1&associITCode={company_code}"
        
        data = {
            'exportFlag': 'false',
            'aggs': '1',
            'associITCode': company_code,
            'classify': '8147,8219,8146,8218,9220,9155,9221,9156,8144,9218,8216,9219,9159,8148,8220,8354,8325,8353,8324,8177',
            'from': '0',
            'scope': '6',
            'size': '50',
            'sort': '1'
        }
        
        try:
            response = requests.post(
                url, 
                headers=self.headers, 
                data=parse.urlencode(data),
                timeout=30
            )
            result = response.json()
            
            if result.get('data', {}).get('hits'):
                for hit in result['data']['hits']:
                    if hit.get('tag', {}).get('label') == '年报':
                        file_info = hit.get('file', [{}])[0]
                        file_name = file_info.get('fileName', '')
                        
                        # 检查是否匹配年报
                        if (re.search(PATTERN_INCLUDE, file_name) and 
                            not re.search(PATTERN_EXCLUDE, file_name) and
                            year in file_name):
                            return file_info.get('fileUrl'), file_name
            return None, None
        except Exception as e:
            print(f"获取年报URL失败: {company_code}, 错误: {e}")
            return None, None
    
    def download_pdf(self, file_url, file_name, folder_path):
        """下载PDF文件"""
        file_name = self.sanitize_filename(file_name)
        file_path = os.path.join(folder_path, f"{file_name}.pdf")
        
        if os.path.exists(file_path):
            print(f"文件已存在: {file_name}")
            return file_path
        
        try:
            response = requests.get(file_url, timeout=60)
            with open(file_path, "wb") as f:
                f.write(response.content)
            print(f"下载成功: {file_name}.pdf")
            return file_path
        except Exception as e:
            print(f"下载失败: {file_name}, 错误: {e}")
            return None
    
    def update_file_record(self, trade_code, file_url, file_name):
        """更新数据库中的文件记录"""
        sql = f"""
        UPDATE {TABLES['annual_report_files']}
        SET fileUrl = :fileUrl, fileName = :fileName, ocr = '已重下'
        WHERE trade_code = :trade_code
        """
        params = {
            "trade_code": trade_code,
            "fileUrl": file_url,
            "fileName": file_name + '.pdf'
        }
        self.db.execute_update(sql, params)

# 初始化爬虫
crawler = AnnualReportCrawler(db_manager)
print("年报爬取模块初始化完成")

In [ ]:
def crawl_annual_reports(companies_df, year='2023', folder_path=None):
    """
    批量爬取年报
    
    参数:
        companies_df: 包含公司信息的DataFrame
        year: 年报年份
        folder_path: PDF保存路径
    """
    if folder_path is None:
        folder_path = PDF_STORAGE_PATH
    
    # 确保目录存在
    os.makedirs(folder_path, exist_ok=True)
    
    success_count = 0
    fail_count = 0
    
    for index, row in companies_df.iterrows():
        print(f"\n--- 处理 {index + 1}/{len(companies_df)} ---")
        
        trade_code = row['trade_code']
        company_name = row.get('company', '')
        
        if not company_name:
            print(f"跳过: {trade_code} - 无公司名称")
            fail_count += 1
            continue
        
        # 搜索公司
        company_code = crawler.search_company(company_name)
        if not company_code:
            print(f"未找到公司: {company_name}")
            fail_count += 1
            sleep(random.uniform(0.5, 1))
            continue
        
        # 获取年报URL
        file_url, file_name = crawler.get_annual_report_url(company_code, year)
        if not file_url:
            print(f"未找到年报: {company_name}")
            fail_count += 1
            sleep(random.uniform(0.5, 1))
            continue
        
        # 下载PDF
        file_path = crawler.download_pdf(file_url, file_name, folder_path)
        if file_path:
            # 更新数据库
            crawler.update_file_record(trade_code, file_url, file_name)
            success_count += 1
        else:
            fail_count += 1
        
        # 随机延迟
        sleep(random.uniform(0.5, 1))
    
    print(f"\n=== 爬取完成 ===")
    print(f"成功: {success_count}, 失败: {fail_count}")
    return success_count, fail_count

print("批量爬取函数定义完成")

<a id='4-pdf处理与ocr'></a>
## 4. PDF处理与OCR

处理PDF文件，使用OCR提取文本内容。

In [ ]:
class PDFProcessor:
    """PDF处理类"""
    
    def __init__(self, db_manager):
        self.db = db_manager
    
    def extract_text_with_pymupdf(self, pdf_path):
        """使用PyMuPDF提取PDF文本"""
        try:
            doc = fitz.open(pdf_path)
            text_parts = []
            
            for page_num in range(doc.page_count):
                page = doc.load_page(page_num)
                text = page.get_text("text")
                text_parts.append(text)
            
            doc.close()
            return '\n'.join(text_parts)
        except Exception as e:
            print(f"PyMuPDF提取失败: {pdf_path}, 错误: {e}")
            return None
    
    def extract_links_from_pdf(self, pdf_path):
        """从PDF中提取超链接"""
        try:
            doc = fitz.open(pdf_path)
            links = []
            
            for page in doc:
                # 提取注释链接
                annots = page.annots()
                if annots:
                    for annot in annots:
                        if annot.type[0] == 1:  # 链接类型
                            links.append(annot.uri)
                
                # 从文本中提取URL
                text = page.get_text("text")
                url_pattern = re.compile(
                    r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
                )
                links.extend(url_pattern.findall(text))
            
            doc.close()
            return list(set(links))
        except Exception as e:
            print(f"提取链接失败: {pdf_path}, 错误: {e}")
            return []
    
    def save_text_file(self, pdf_path, text):
        """保存提取的文本到文件"""
        txt_path = pdf_path.replace('.pdf', '.txt')
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(text)
        return txt_path
    
    def update_ocr_status(self, file_name, status):
        """更新OCR状态"""
        sql = f"""
        UPDATE {TABLES['annual_report_files']}
        SET ocr = :status
        WHERE fileName = :fileName
        """
        params = {
            "fileName": file_name,
            "status": status
        }
        self.db.execute_update(sql, params)

# 初始化PDF处理器
pdf_processor = PDFProcessor(db_manager)
print("PDF处理模块初始化完成")

In [ ]:
def process_pdfs_batch(folder_path=None, limit=None):
    """
    批量处理PDF文件
    
    参数:
        folder_path: PDF文件所在目录
        limit: 处理数量限制，None表示不限制
    """
    if folder_path is None:
        folder_path = PDF_STORAGE_PATH
    
    # 获取所有PDF文件
    pdf_files = [f for f in os.listdir(folder_path) if f.endswith('.pdf')]
    
    if limit:
        pdf_files = pdf_files[:limit]
    
    success_count = 0
    fail_count = 0
    skip_count = 0
    
    for i, filename in enumerate(pdf_files):
        print(f"\n--- 处理 {i + 1}/{len(pdf_files)}: {filename} ---")
        
        pdf_path = os.path.join(folder_path, filename)
        txt_path = pdf_path.replace('.pdf', '.txt')
        
        # 检查是否已处理
        if os.path.exists(txt_path):
            print("已存在OCR结果，跳过")
            skip_count += 1
            continue
        
        # 提取文本
        text = pdf_processor.extract_text_with_pymupdf(pdf_path)
        
        if text:
            # 保存文本
            pdf_processor.save_text_file(pdf_path, text)
            # 更新状态
            pdf_processor.update_ocr_status(filename, datetime.now().strftime('%Y-%m-%d'))
            success_count += 1
            print(f"OCR完成，文本长度: {len(text)}")
        else:
            pdf_processor.update_ocr_status(filename, '失败')
            fail_count += 1
        
        # 延迟
        sleep(0.1)
    
    print(f"\n=== 处理完成 ===")
    print(f"成功: {success_count}, 失败: {fail_count}, 跳过: {skip_count}")
    return success_count, fail_count, skip_count

print("批量OCR处理函数定义完成")

In [ ]:
def check_ocr_status():
    """检查OCR完成状态"""
    sql = f"""
    SELECT fileName, ocr
    FROM {TABLES['annual_report_files']}
    WHERE fileName != ''
    """
    df = db_manager.read_sql(sql)
    
    total = len(df)
    ocr_done = len(df[df['ocr'] != ''])
    ocr_pending = len(df[df['ocr'] == ''])
    
    print(f"总文件数: {total}")
    print(f"已完成OCR: {ocr_done}")
    print(f"待OCR: {ocr_pending}")
    print(f"完成率: {ocr_done / total * 100:.2f}%")
    
    return {
        'total': total,
        'done': ocr_done,
        'pending': ocr_pending,
        'rate': ocr_done / total * 100 if total > 0 else 0
    }

# 检查OCR状态
ocr_status = check_ocr_status()

<a id='5-大模型信息提取'></a>
## 5. 大模型信息提取

使用通义千问大模型从OCR结果中提取关键财务信息。

In [ ]:
class LLMExtractor:
    """大模型信息提取类"""
    
    def __init__(self):
        self.client = OpenAI(
            api_key=QWEN_API_KEY,
            base_url=QWEN_BASE_URL
        )
        self.model = QWEN_MODEL
        self.extraction_prompt = CONFIG['extraction_prompt']
    
    def upload_file(self, file_path):
        """上传文件到通义千问"""
        try:
            file_object = self.client.files.create(
                file=Path(file_path),
                purpose="file-extract"
            )
            return file_object.id
        except Exception as e:
            print(f"上传文件失败: {file_path}, 错误: {e}")
            return None
    
    def wait_for_processing(self, file_id, timeout=300):
        """等待文件处理完成"""
        start_time = time.time()
        
        while time.time() - start_time < timeout:
            try:
                files_list = self.client.files.list()
                for item in files_list.data:
                    if item.id == file_id:
                        if item.status == 'processed':
                            return True
                        elif item.status == 'failed':
                            return False
                time.sleep(10)
            except Exception as e:
                print(f"检查文件状态失败: {e}")
                time.sleep(5)
        
        return False
    
    def extract_financial_info(self, file_id):
        """使用大模型提取财务信息"""
        try:
            completion = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        'role': 'system',
                        'content': 'You are a helpful assistant.'
                    },
                    {
                        'role': 'system',
                        'content': f'fileid://{file_id}'
                    },
                    {
                        'role': 'user',
                        'content': self.extraction_prompt
                    }
                ],
                stream=False
            )
            
            return completion.choices[0].message.content.strip()
        except Exception as e:
            print(f"提取信息失败: {file_id}, 错误: {e}")
            return None
    
    def parse_extraction_result(self, result_text):
        """解析提取结果"""
        results = []
        
        lines = result_text.strip().split('\n')
        if not lines:
            return results
        
        # 获取表头
        header = lines[0].split('|')
        
        # 解析数据行
        for line in lines[1:]:
            parts = line.split('|')
            if len(parts) == len(header):
                data = dict(zip(header, parts))
                results.append(data)
        
        return results

# 初始化大模型提取器
llm_extractor = LLMExtractor()
print("大模型提取模块初始化完成")

In [ ]:
def process_file_with_llm(file_path, db_manager):
    """
    使用大模型处理单个文件
    
    参数:
        file_path: 文件路径
        db_manager: 数据库管理器
    """
    filename = os.path.basename(file_path)
    print(f"处理文件: {filename}")
    
    # 上传文件
    file_id = llm_extractor.upload_file(file_path)
    if not file_id:
        return None
    
    print(f"文件已上传，ID: {file_id}")
    
    # 等待处理
    if not llm_extractor.wait_for_processing(file_id):
        print("文件处理超时或失败")
        return None
    
    # 提取信息
    result = llm_extractor.extract_financial_info(file_id)
    if not result:
        return None
    
    # 解析结果
    parsed_results = llm_extractor.parse_extraction_result(result)
    
    # 存储结果
    for data in parsed_results:
        data['filename'] = filename
        
        sql = f"""
        INSERT INTO {TABLES['extraction_results']} 
        (filename, 年份, 项目, 增加原因, 金额, 原始金额, 原始金额单位)
        VALUES (:filename, :年份, :项目, :增加原因, :金额, :原始金额, :原始金额单位)
        """
        
        try:
            db_manager.execute_update(sql, data)
        except Exception as e:
            print(f"存储结果失败: {e}")
    
    print(f"提取完成，共 {len(parsed_results)} 条记录")
    return parsed_results

print("大模型处理函数定义完成")

In [ ]:
def batch_process_with_llm(folder_path=None, limit=None):
    """
    批量使用大模型处理文件
    
    参数:
        folder_path: 文件所在目录
        limit: 处理数量限制
    """
    if folder_path is None:
        folder_path = PDF_STORAGE_PATH
    
    # 获取所有txt文件（OCR结果）
    txt_files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    
    if limit:
        txt_files = txt_files[:limit]
    
    success_count = 0
    fail_count = 0
    
    for i, filename in enumerate(txt_files):
        print(f"\n--- 处理 {i + 1}/{len(txt_files)}: {filename} ---")
        
        file_path = os.path.join(folder_path, filename)
        
        try:
            result = process_file_with_llm(file_path, db_manager)
            if result:
                success_count += 1
            else:
                fail_count += 1
        except Exception as e:
            print(f"处理失败: {e}")
            fail_count += 1
        
        # 延迟
        sleep(1)
    
    print(f"\n=== 处理完成 ===")
    print(f"成功: {success_count}, 失败: {fail_count}")
    return success_count, fail_count

print("批量大模型处理函数定义完成")

<a id='6-数据处理与存储'></a>
## 6. 数据处理与存储

数据处理、转换和存储相关功能。

In [ ]:
class DataProcessor:
    """数据处理类"""
    
    def __init__(self, db_manager):
        self.db = db_manager
    
    @staticmethod
    def convert_amount_to_yi(amount, unit):
        """
        将金额转换为亿元
        
        参数:
            amount: 原始金额
            unit: 原始单位
        """
        try:
            amount = float(amount.replace(',', ''))
        except:
            return None
        
        converters = {
            '元': 1e-8,
            '万元': 1e-4,
            '百万': 1e-2,
            '千元': 1e-5,
            '亿': 1
        }
        
        return amount * converters.get(unit, 1)
    
    def get_extraction_summary(self):
        """获取提取结果汇总"""
        sql = f"""
        SELECT 项目, 年份, COUNT(*) as 数量, SUM(金额) as 总金额
        FROM {TABLES['extraction_results']}
        GROUP BY 项目, 年份
        ORDER BY 年份, 项目
        """
        return self.db.read_sql(sql)
    
    def get_company_extraction_results(self, filename):
        """获取指定公司的提取结果"""
        sql = f"""
        SELECT * FROM {TABLES['extraction_results']}
        WHERE filename = :filename
        """
        return self.db.read_sql_with_params(sql, {'filename': filename})
    
    def export_to_excel(self, output_path):
        """导出提取结果到Excel"""
        sql = f"SELECT * FROM {TABLES['extraction_results']}"
        df = self.db.read_sql(sql)
        df.to_excel(output_path, index=False)
        print(f"已导出到: {output_path}")
        return output_path

# 初始化数据处理器
data_processor = DataProcessor(db_manager)
print("数据处理模块初始化完成")

In [ ]:
def extract_capital_reserve_info(pdf_path, db_manager):
    """
    从PDF中提取资本公积相关信息
    
    参数:
        pdf_path: PDF文件路径
        db_manager: 数据库管理器
    """
    keywords = ["实施资本", "资本公积"]
    extended_keywords = ["划转", "划入", "投入", "拨入", "转增", "注入", 
                        "注资", "出资", "转入", "拨款", "豁免", "免除", 
                        "无偿", "捐赠", "增资"]
    
    try:
        doc = fitz.open(pdf_path)
        results = []
        
        for page_num in range(doc.page_count):
            page = doc.load_page(page_num)
            text = page.get_text("text")
            paragraphs = text.split('\n\n')
            
            for paragraph in paragraphs:
                if (any(kw in paragraph for kw in keywords) and 
                    any(ext_kw in paragraph for ext_kw in extended_keywords)):
                    results.append(paragraph)
        
        doc.close()
        
        # 存储结果
        file_name = os.path.basename(pdf_path)
        for paragraph in results:
            sql = f"""
            INSERT IGNORE INTO {TABLES['capital_reserve']} (file_name, paragraph)
            VALUES (:file_name, :paragraph)
            """
            params = {
                "file_name": file_name,
                "paragraph": paragraph
            }
            db_manager.execute_update(sql, params)
        
        return results
    except Exception as e:
        print(f"提取资本公积信息失败: {e}")
        return []

print("资本公积提取函数定义完成")

In [ ]:
def generate_report(output_path=None):
    """生成年报处理汇总报告"""
    if output_path is None:
        output_path = os.path.join(PDF_STORAGE_PATH, '年报处理报告.xlsx')
    
    # 获取各项统计数据
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        # 1. 文件处理状态
        sql_files = f"""
        SELECT 
            COUNT(*) as 总数,
            SUM(CASE WHEN fileName != '' THEN 1 ELSE 0 END) as 已下载,
            SUM(CASE WHEN ocr != '' AND ocr NOT IN ('失败', '未ocr', '已重下') THEN 1 ELSE 0 END) as 已OCR
        FROM {TABLES['annual_report_files']}
        """
        df_status = db_manager.read_sql(sql_files)
        df_status.to_excel(writer, sheet_name='处理状态', index=False)
        
        # 2. 提取结果汇总
        df_summary = data_processor.get_extraction_summary()
        df_summary.to_excel(writer, sheet_name='提取汇总', index=False)
        
        # 3. 详细提取结果
        sql_detail = f"SELECT * FROM {TABLES['extraction_results']}"
        df_detail = db_manager.read_sql(sql_detail)
        df_detail.to_excel(writer, sheet_name='提取详情', index=False)
    
    print(f"报告已生成: {output_path}")
    return output_path

print("报告生成函数定义完成")

<a id='7-主流程执行'></a>
## 7. 主流程执行

执行完整的年报处理流程。

In [ ]:
def run_full_pipeline(year='2023', crawl=True, ocr=True, extract=True, report=True):
    """
    执行完整的年报处理流程
    
    参数:
        year: 处理的年报年份
        crawl: 是否执行爬取
        ocr: 是否执行OCR
        extract: 是否执行大模型提取
        report: 是否生成报告
    """
    print(f"="*50)
    print(f"开始执行年报处理流程 - {year}年度")
    print(f"="*50)
    
    results = {}
    
    # 1. 爬取年报
    if crawl:
        print("\n[步骤1] 爬取年报PDF...")
        df_files = query_annual_report_files()
        # 只处理未下载的文件
        df_pending = df_files[df_files['fileName'] == '']
        if len(df_pending) > 0:
            success, fail = crawl_annual_reports(df_pending, year)
            results['crawl'] = {'success': success, 'fail': fail}
        else:
            print("所有文件已下载，跳过爬取")
            results['crawl'] = {'success': 0, 'fail': 0}
    
    # 2. OCR处理
    if ocr:
        print("\n[步骤2] OCR处理PDF...")
        success, fail, skip = process_pdfs_batch()
        results['ocr'] = {'success': success, 'fail': fail, 'skip': skip}
    
    # 3. 大模型提取
    if extract:
        print("\n[步骤3] 大模型提取财务信息...")
        success, fail = batch_process_with_llm()
        results['extract'] = {'success': success, 'fail': fail}
    
    # 4. 生成报告
    if report:
        print("\n[步骤4] 生成汇总报告...")
        report_path = generate_report()
        results['report'] = report_path
    
    print(f"\n" + "="*50)
    print("年报处理流程完成")
    print("="*50)
    
    return results

print("主流程函数定义完成")

In [ ]:
# 执行主流程
# 取消注释以下代码来执行完整流程

# results = run_full_pipeline(
#     year='2023',
#     crawl=False,  # 需要时设为True
#     ocr=True,
#     extract=True,
#     report=True
# )

print("如需执行完整流程，请取消上面的注释")

In [ ]:
# 单独执行各步骤的示例

# 检查当前状态
print("当前处理状态:")
status = check_ocr_status()
print()

# 查看提取结果汇总
print("提取结果汇总:")
try:
    summary = data_processor.get_extraction_summary()
    print(summary)
except Exception as e:
    print(f"获取汇总失败: {e}")

In [ ]:
# 关闭数据库连接
# db_manager.yq_engine.dispose()
# db_manager.timinfo_engine.dispose()
# print("数据库连接已关闭")